# Model 2: Pretrained BERT with AutoModelForMultipleChoice

**Project:** Smart MCQ Solver · DL & GenAI · Milestone 3  
**Kernel path:** `nb/pretrained/pre_trained.ipynb`

---

## Architecture

Uses HuggingFace `AutoModelForMultipleChoice` which natively handles all 5 options simultaneously in a single forward pass.

| Step | Description |
|------|-------------|
| **Tokenization** | Each question is paired with each of its 5 options: `[CLS] prompt [SEP] option_text [SEP]` |
| **Input shape** | `(batch, 5, seq_len)` — all 5 candidate options stacked along dim-1 |
| **Encoder** | `bert-base-uncased` — 12 transformer layers, 768 hidden dim |
| **Classifier** | Single linear head over `[CLS]` token produces 5 logits — one per option |
| **Loss** | `CrossEntropyLoss` over the 5-way class distribution — the ground truth is the correct option index |
| **Inference** | `torch.topk(logits, 3)` ranks the top-3 options; written to `submission.csv` |

> **Viva Defence:** Unlike pairwise binary scoring (5 independent forward passes), `AutoModelForMultipleChoice`
> processes all 5 options in one pass with shared attention across the prompt context. This is architecturally
> closer to how BERT was originally evaluated on SWAG/CommonsenseQA tasks and is both faster and more accurate.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print('Imports ready.')

In [ ]:
# Clone the model locally to avoid Kaggle download timeouts.
# If the directory already exists (cached from a previous run), skip it.
MODEL_DIR = './bert-base-uncased'

if not os.path.exists(MODEL_DIR):
    print('Cloning bert-base-uncased from HuggingFace Hub...')
    os.system('git clone https://huggingface.co/bert-base-uncased')
    print('Clone complete.')
else:
    print(f'Model directory already exists at {MODEL_DIR} — skipping clone.')

print(os.listdir(MODEL_DIR) if os.path.exists(MODEL_DIR) else 'Clone may have failed.')

In [ ]:
KAGGLE_INPUT = '/kaggle/input/competitions/smart-mcq-solver-challenge'

def get_path(filename: str) -> str:
    """Resolve data files across local dev and Kaggle cloud environments.

    Resolution order:
      1. ../../data/   — local when CWD is nb/pretrained/
      2. data/         — local when CWD is project root
      3. Kaggle mount  — production GPU kernel
    """
    candidates = [
        os.path.join('..', '..', 'data', filename),
        os.path.join('data', filename),
        os.path.join(KAGGLE_INPUT, filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return filename  # last-ditch: let pandas raise the error with context

# Device: GPU on Kaggle, CPU locally for smoke tests
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device : {DEVICE}')

In [ ]:
# Authenticate W&B — runs land in project: 23f2004343-t22026
wandb.login(
    key=os.environ.get("WANDB_API_KEY", ""),
    relogin=True,
)

run = wandb.init(
    project='23f2004343-t22026',
    name='Model_2_BERT_MultipleChoice',
    config={
        'base_model'     : 'bert-base-uncased',
        'max_seq_length' : 128,
        'batch_size'     : 8,
        'epochs'         : 3,
        'learning_rate'  : 2e-5,
        'warmup_ratio'   : 0.1,
        'weight_decay'   : 0.01,
        'optimizer'      : 'AdamW',
        'loss_fn'        : 'CrossEntropyLoss',
        'num_options'    : 5,
    },
)
cfg = wandb.config
print(f'W&B run : {run.name}  |  project : {run.project}')
print(f'Config  : {dict(cfg)}')

In [ ]:
trn_df = pd.read_csv(get_path('train.csv'))
tst_df = pd.read_csv(get_path('test.csv'))
print(f'train : {trn_df.shape}  |  test : {tst_df.shape}')
print(f'columns : {list(trn_df.columns)}')

OPTIONS = ['A', 'B', 'C', 'D', 'E']

# Load tokenizer from the locally cloned directory first, fall back to hub name
MODEL_NAME = MODEL_DIR if os.path.exists(MODEL_DIR) else cfg.base_model
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN    = int(cfg.max_seq_length)


class MCQMultiChoiceDataset(Dataset):
    """Tokenizes each question into 5 (prompt, option) pairs stacked on dim-1.

    Output per item:
      input_ids      : (5, max_len)  — one row per option
      attention_mask : (5, max_len)
      token_type_ids : (5, max_len)  — segment IDs needed by BERT
      label          : int in [0, 4] — index of the correct option (train only)

    Viva Defence: Stacking all 5 sequences lets AutoModelForMultipleChoice
    compute cross-attention over the full option set in a single forward pass,
    rather than running 5 separate passes as pairwise scoring does.
    """

    def __init__(self, df: pd.DataFrame, is_train: bool = True):
        self.df       = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])

        # Tokenize each (prompt, option) pair
        encodings = tokenizer(
            [prompt] * 5,
            [str(row[opt]) for opt in OPTIONS],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        # encodings tensors: (5, max_len)
        item = {
            'input_ids'      : encodings['input_ids'],
            'attention_mask' : encodings['attention_mask'],
            'token_type_ids' : encodings.get('token_type_ids', torch.zeros(5, MAX_LEN, dtype=torch.long)),
        }

        if self.is_train:
            ans   = str(row['answer']).strip().upper()
            label = OPTIONS.index(ans) if ans in OPTIONS else 0
            item['label'] = torch.tensor(label, dtype=torch.long)

        return item


print('Dataset class defined.')

In [ ]:
trn_sub, val_sub = train_test_split(trn_df, test_size=0.1, random_state=SEED)

BATCH = int(cfg.batch_size)

trn_loader = DataLoader(
    MCQMultiChoiceDataset(trn_sub, is_train=True),
    batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    MCQMultiChoiceDataset(val_sub, is_train=True),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True,
)
tst_loader = DataLoader(
    MCQMultiChoiceDataset(tst_df, is_train=False),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True,
)

probe = next(iter(trn_loader))
print(f"input_ids shape : {probe['input_ids'].shape}   # (batch, 5, max_len)")
print(f"labels          : {probe['label'].tolist()[:8]}")

In [ ]:
# AutoModelForMultipleChoice expects (batch, num_choices, seq_len) inputs
# and returns logits of shape (batch, num_choices).
# Internally it runs BERT once per option (parallelized in the forward pass)
# and applies a linear classifier on the [CLS] representations.
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model  : {cfg.base_model}  |  trainable params : {n_params:,}')

criterion = nn.CrossEntropyLoss()  # labels are class indices, not one-hot
optimizer = AdamW(
    model.parameters(),
    lr=float(cfg.learning_rate),
    weight_decay=float(cfg.weight_decay),
    eps=1e-8,
)

total_steps  = len(trn_loader) * int(cfg.epochs)
warmup_steps = int(total_steps * float(cfg.warmup_ratio))
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f'Steps : {total_steps}  |  warmup : {warmup_steps}')

In [ ]:
def run_eval(model, loader):
    """Validation pass — returns loss, accuracy, macro-F1.

    Viva Defence:
      - torch.argmax(logits, dim=1) converts 5-class logits to the index of
        the highest-scoring option — no sigmoid or threshold needed.
      - Macro-F1 treats each option class equally regardless of class frequency.
    """
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            masks = batch['attention_mask'].to(DEVICE)
            ttype = batch['token_type_ids'].to(DEVICE)
            labs  = batch['label'].to(DEVICE)

            out    = model(input_ids=ids, attention_mask=masks, token_type_ids=ttype)
            logits = out.logits  # (batch, 5)
            loss   = criterion(logits, labs)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labs.cpu().tolist())

    n = max(len(loader), 1)
    return {
        'val_loss'     : total_loss / n,
        'val_accuracy' : accuracy_score(all_labels, all_preds),
        'val_f1_macro' : f1_score(all_labels, all_preds, average='macro', zero_division=0),
    }


best_val_loss = float('inf')
EPOCHS        = int(cfg.epochs)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch in trn_loader:
        ids   = batch['input_ids'].to(DEVICE)
        masks = batch['attention_mask'].to(DEVICE)
        ttype = batch['token_type_ids'].to(DEVICE)
        labs  = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=masks, token_type_ids=ttype).logits
        loss   = criterion(logits, labs)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent gradient explosion
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    avg_trn = running_loss / max(len(trn_loader), 1)
    val_m   = run_eval(model, val_loader)

    wandb.log({'epoch': epoch, 'train_loss': avg_trn, **val_m})
    print(
        f'Epoch {epoch}/{EPOCHS}  '
        f'trn_loss={avg_trn:.4f}  '
        f'val_loss={val_m["val_loss"]:.4f}  '
        f'val_acc={val_m["val_accuracy"]:.4f}  '
        f'val_f1={val_m["val_f1_macro"]:.4f}'
    )

    # Save checkpoint on improvement
    if val_m['val_loss'] < best_val_loss:
        best_val_loss = val_m['val_loss']
        torch.save(model.state_dict(), '/kaggle/working/bert_mcq_best.pt')
        print(f'  ✓ Checkpoint saved (val_loss={best_val_loss:.4f})')

wandb.run.summary['best_val_loss'] = best_val_loss
print('\nTraining complete.')

In [ ]:
# Load best checkpoint
ckpt = '/kaggle/working/bert_mcq_best.pt'
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print(f'Loaded best checkpoint: {ckpt}')

model.eval()
OPTIONS_ARR = np.array(OPTIONS)
all_top3    = []

print(f'Running batched inference on {len(tst_df)} test questions...')
with torch.no_grad():
    for batch in tst_loader:
        ids   = batch['input_ids'].to(DEVICE)
        masks = batch['attention_mask'].to(DEVICE)
        ttype = batch['token_type_ids'].to(DEVICE)

        logits   = model(input_ids=ids, attention_mask=masks, token_type_ids=ttype).logits  # (B, 5)
        top3_idx = torch.topk(logits, 3, dim=1).indices.cpu().numpy()  # (B, 3) — descending

        for row_idx in top3_idx:
            all_top3.append(' '.join(OPTIONS_ARR[row_idx]))

# Align output with sample_submission.csv to guarantee no KeyError
sub_template = pd.read_csv(get_path('sample_submission.csv'))
print(f'sample_submission columns : {list(sub_template.columns)}')
print(f'Predictions generated     : {len(all_top3)}')

sub_df = pd.DataFrame({
    sub_template.columns[0]: tst_df['id'].values,
    sub_template.columns[1]: all_top3,
})

sub_df.to_csv('submission.csv', index=False)
print(f'submission.csv written — {len(sub_df)} rows')
print(sub_df.head())

# W&B artifact + close
art = wandb.Artifact('submission_bert_mcq', type='predictions')
art.add_file('submission.csv')
wandb.log_artifact(art)
wandb.finish()
print('W&B run closed.')